# 📚 Notebook 03 — 风险指标深度解析

**第一阶段：基础知识** · 前置要求：Notebook 02（指标，对数收益率）

---

## 🎯 学习目标

完成本 Notebook 后，你将能够：

1. 从原始收益率计算**夏普比率**、**索提诺比率**和**卡尔玛比率**
2. 理解机器人为何使用**复合评分**：索提诺 40% + 夏普 30% + 卡尔玛 30%
3. 计算**最大回撤**和权益曲线
4. 使用 **Delta 方法**推导风险比率的**标准误差**
5. 评估回测结果是否具有统计显著性

In [ ]:
# ── 环境设置 ──
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from datetime import datetime, timezone, timedelta
import requests

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

# ── 生成样本收益率 ──
def fetch_btc_hourly(days: int = 90) -> pd.DataFrame:
    end_ms = int(datetime.now(timezone.utc).timestamp() * 1000)
    start_ms = end_ms - days * 86_400_000
    all_rows = []
    cursor = start_ms
    while cursor < end_ms:
        resp = requests.get("https://api.binance.com/api/v3/klines",
                           params={"symbol": "BTCUSDT", "interval": "1h",
                                   "startTime": cursor, "endTime": end_ms, "limit": 1000},
                           timeout=10)
        resp.raise_for_status()
        rows = resp.json()
        if not rows: break
        all_rows.extend(rows)
        cursor = int(rows[-1][0]) + 3_600_000
        if len(rows) < 1000: break
    df = pd.DataFrame(all_rows, columns=["open_time","open","high","low","close","volume",
                                          "close_time","quote_vol","trades","taker_base","taker_quote","ignore"])
    for col in ["open","high","low","close","volume"]:
        df[col] = df[col].astype(float)
    df["timestamp"] = pd.to_datetime(df["open_time"], unit="ms", utc=True)
    df = df.set_index("timestamp")[["open","high","low","close","volume"]]
    return df

df = fetch_btc_hourly(90)
close = df["close"]
log_returns = np.log(close / close.shift(1)).dropna()
print(f"✅ 已加载 {len(log_returns)} 个小时对数收益率")
print(f"   均值: {log_returns.mean():.6f}  标准差: {log_returns.std():.6f}")

---

## 📐 第一节：夏普比率 (Sharpe Ratio)

夏普比率衡量**每单位总风险的收益**：

$$\text{Sharpe} = \frac{\bar{r} - r_f}{\sigma}$$

其中：
- $\bar{r}$ = 每期平均收益率
- $r_f$ = 每期无风险利率（加密货币中使用 0）
- $\sigma$ = 收益率标准差

### 年化处理

对于小时数据，每年 $N = 8{,}760$ 小时：

$$\text{Sharpe}_{\text{年化}} = \text{Sharpe}_{\text{小时}} \times \sqrt{N} = \frac{\bar{r}}{\sigma} \times \sqrt{8760}$$

### 解读标准

| 夏普比率 | 质量 |
|---------|------|
| < 0 | 亏损 |
| 0–1 | 低于平均 |
| 1–2 | 良好 |
| 2–3 | 非常好 |
| > 3 | 卓越（或过拟合！）|

In [ ]:
# ── 从零计算夏普比率 ──

def sharpe_ratio(returns: pd.Series, periods_per_year: int = 8760) -> float:
    """年化夏普比率（超额收益 / 波动率）。
    
    用于机器人的复合评分中：
    索提诺 (40%) + 夏普 (30%) + 卡尔玛 (30%)
    """
    if returns.std() == 0:
        return 0.0
    return (returns.mean() / returns.std()) * np.sqrt(periods_per_year)

sharpe = sharpe_ratio(log_returns)
print(f"年化夏普比率: {sharpe:.4f}")
print(f"\n分解计算:")
print(f"  小时平均收益: {log_returns.mean():.8f}")
print(f"  小时波动率:   {log_returns.std():.8f}")
print(f"  小时夏普:     {log_returns.mean() / log_returns.std():.6f}")
print(f"  × √8760 =    {sharpe:.4f}")

---

## 📐 第二节：索提诺比率 — 只惩罚下行风险

夏普比率对**所有**波动率一视同仁 — 但上行波动率是好事！索提诺比率只惩罚**下行偏差**：

$$\text{Sortino} = \frac{\bar{r} - r_f}{\sigma_d}$$

其中**下行偏差**为：

$$\sigma_d = \sqrt{\frac{1}{n} \sum_{t=1}^{n} \min(r_t - r_f, 0)^2}$$

### 为什么索提诺权重最高（40%）

机器人将索提诺作为**首要**指标（40% 权重），因为：
- 加密货币收益具有高度不对称性（偶尔出现大幅上涨）
- 我们不希望惩罚大幅上涨行情
- 下行风险才是真正会让账户爆仓的因素

In [ ]:
# ── 从零计算索提诺比率 ──

def sortino_ratio(returns: pd.Series, periods_per_year: int = 8760) -> float:
    """年化索提诺比率（超额收益 / 下行偏差）。
    
    这是复合评分中权重最高的指标（40%）。
    """
    # 下行偏差：只计算负收益的标准差
    downside = returns.clip(upper=0)  # 只保留负收益
    downside_dev = np.sqrt((downside ** 2).mean())  # 下行均方根
    
    if downside_dev == 0:
        return 0.0
    
    return (returns.mean() / downside_dev) * np.sqrt(periods_per_year)

sortino = sortino_ratio(log_returns)
print(f"年化索提诺比率: {sortino:.4f}")
print(f"年化夏普比率:   {sharpe:.4f}")
print(f"\n索提诺 / 夏普 = {sortino / sharpe:.2f}x" if sharpe != 0 else "")
print(f"（正收益波动性大于负收益时，索提诺 > 夏普）")

---

## 📐 第三节：最大回撤与卡尔玛比率

### 最大回撤 (Maximum Drawdown, MaxDD)

**最大回撤**是累计收益中最大的峰谷跌幅：

$$\text{MaxDD} = \max_{t} \left( \frac{\text{Peak}_t - \text{Value}_t}{\text{Peak}_t} \right)$$

### 卡尔玛比率 (Calmar Ratio)

$$\text{Calmar} = \frac{\text{年化收益率}}{|\text{MaxDD}|}$$

卡尔玛比率回答的问题是：*
1
*

它在复合评分中获得 **30% 权重**，因为它捕捉了夏普和索提诺遗漏的**尾部风险**。

In [ ]:
# ── 最大回撤和卡尔玛比率 ──

def compute_drawdown(returns: pd.Series) -> pd.DataFrame:
    """从收益率计算回撤序列。"""
    # 累计财富（权益曲线）
    cumulative = (1 + returns).cumprod()
    
    # 滚动最高点
    peak = cumulative.cummax()
    
    # 回撤：距离最高点的距离
    drawdown = (cumulative - peak) / peak
    
    return pd.DataFrame({
        'cumulative': cumulative,
        'peak': peak,
        'drawdown': drawdown,
    })

def max_drawdown(returns: pd.Series) -> float:
    """最大回撤（正值分数，如 0.15 = 15% 回撤）。"""
    dd_df = compute_drawdown(returns)
    return abs(dd_df['drawdown'].min())

def calmar_ratio(returns: pd.Series, periods_per_year: int = 8760) -> float:
    """年化卡尔玛比率（年化收益 / 最大回撤）。
    
    在复合评分中占 30% 权重。
    """
    ann_return = returns.mean() * periods_per_year
    mdd = max_drawdown(returns)
    if mdd == 0:
        return 0.0
    return ann_return / mdd

mdd = max_drawdown(log_returns)
calmar = calmar_ratio(log_returns)

print(f"最大回撤:   {mdd:.2%}")
print(f"卡尔玛比率: {calmar:.4f}")

In [ ]:
# ── 权益曲线和回撤可视化 ──
dd_df = compute_drawdown(log_returns)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 8), height_ratios=[2, 1], sharex=True)

# 权益曲线
ax1.plot(dd_df['cumulative'].index, dd_df['cumulative'], label='累计收益', color='#2196F3', linewidth=1.5)
ax1.plot(dd_df['peak'].index, dd_df['peak'], label='历史高点', color='gray', linewidth=1, linestyle='--', alpha=0.5)
ax1.set_ylabel('权益（起始值 1.0）')
ax1.set_title('BTC/USDT — 权益曲线与回撤', fontsize=14)
ax1.legend()

# 回撤
ax2.fill_between(dd_df['drawdown'].index, 0, dd_df['drawdown'], color='red', alpha=0.3)
ax2.plot(dd_df['drawdown'].index, dd_df['drawdown'], color='darkred', linewidth=1)
ax2.set_ylabel('回撤')
ax2.set_title(f'回撤图（最大: {mdd:.2%}）', fontsize=11)

# 标记最大回撤点
worst_idx = dd_df['drawdown'].idxmin()
ax2.annotate(f'最大回撤: {mdd:.2%}', xy=(worst_idx, dd_df['drawdown'].min()),
             fontsize=11, color='darkred', fontweight='bold',
             xytext=(10, -30), textcoords='offset points',
             arrowprops=dict(arrowstyle='->', color='darkred'))

plt.tight_layout()
plt.show()

---

## 📐 第四节：复合评分

机器人使用**加权复合评分**评估策略表现：

$$\text{Score} = 0.40 \times \text{Sortino} + 0.30 \times \text{Sharpe} + 0.30 \times \text{Calmar}$$

### 为什么选择这些权重？

| 指标 | 权重 | 捕捉 | 弱点 |
|------|------|------|------|
| 索提诺 | 40% | 下行风险 | 忽略相关性结构 |
| 夏普 | 30% | 总风险调整收益 | 惩罚上行波动 |
| 卡尔玛 | 30% | 尾部/回撤风险 | 单点估计 |

In [ ]:
# ── 复合评分 ──

def composite_score(returns: pd.Series, periods_per_year: int = 8760) -> dict:
    """计算机器人的复合绩效评分。
    
    权重: 索提诺 40% + 夏普 30% + 卡尔玛 30%
    参考: bot/backtest/core_module_backtester.py
    """
    s = sharpe_ratio(returns, periods_per_year)
    so = sortino_ratio(returns, periods_per_year)
    c = calmar_ratio(returns, periods_per_year)
    composite = 0.40 * so + 0.30 * s + 0.30 * c
    
    return {
        'sharpe': s,
        'sortino': so,
        'calmar': c,
        'composite': composite,
        'max_drawdown': max_drawdown(returns),
        'annual_return': returns.mean() * periods_per_year,
    }

scores = composite_score(log_returns)
print("═" * 50)
print("         复合绩效评分卡")
print("═" * 50)
print(f"  夏普   (30%):  {scores['sharpe']:>8.4f}")
print(f"  索提诺 (40%):  {scores['sortino']:>8.4f}")
print(f"  卡尔玛 (30%):  {scores['calmar']:>8.4f}")
print(f"  ─────────────────────────────")
print(f"  复合评分:      {scores['composite']:>8.4f}")
print(f"")
print(f"  年化收益:      {scores['annual_return']:>8.2%}")
print(f"  最大回撤:      {scores['max_drawdown']:>8.2%}")
print("═" * 50)

---

## 📐 第五节：Delta 方法计算标准误差

夏普比率为 1.5 并无意义，如果**不确定性**为 ±2.0。我们需要**标准误差**来评估统计显著性。

### Delta 方法

对于样本统计量的函数 $g(\mu, \sigma)$，Delta 方法近似其方差：

$$\text{Var}[g(\hat{\mu}, \hat{\sigma})] \approx \nabla g^T \cdot \Sigma \cdot \nabla g$$

### 夏普比率标准误差

对于 $\text{SR} = \mu / \sigma$，在正态性假设下的 SE 近似为：

$$\text{SE}(\text{SR}) \approx \sqrt{\frac{1 + \frac{\text{SR}^2}{2}}{n}}$$

这同时考虑了 $\mu$ 和 $\sigma$ 的估计误差。

### 索提诺比率标准误差

索提诺的 SE 更复杂，因为下行偏差只使用负收益率：

$$\text{SE}(\text{Sortino}) \approx \frac{1}{\sigma_d} \sqrt{\frac{\sigma^2}{n} + \frac{\mu^2 \cdot \text{Var}(\sigma_d)}{\sigma_d^2}}$$

我们在策略笔记中用矩阵微积分推导了这个公式。

In [ ]:
# ── Delta 方法标准误差 ──

def sharpe_standard_error(returns: pd.Series, periods_per_year: int = 8760) -> dict:
    """使用 Delta 方法计算夏普比率及其标准误差。
    
    参考: strategy_notes_ZH.md 第5节（Delta 方法推导）
    """
    n = len(returns)
    mu = returns.mean()
    sigma = returns.std()
    
    if sigma == 0:
        return {'sharpe': 0.0, 'se': 0.0, 't_stat': 0.0, 'p_value': 1.0}
    
    # 每期夏普
    sr = mu / sigma
    
    # Delta 方法 SE: sqrt((1 + SR²/2) / n)
    se_per_period = np.sqrt((1 + sr**2 / 2) / n)
    
    # 年化两者
    sr_ann = sr * np.sqrt(periods_per_year)
    se_ann = se_per_period * np.sqrt(periods_per_year)
    
    # t 统计量：夏普是否显著不同于 0？
    t_stat = sr / se_per_period
    p_value = 2 * (1 - stats.norm.cdf(abs(t_stat)))
    
    return {
        'sharpe': sr_ann,
        'se': se_ann,
        'ci_lower': sr_ann - 1.96 * se_ann,
        'ci_upper': sr_ann + 1.96 * se_ann,
        't_stat': t_stat,
        'p_value': p_value,
    }

result = sharpe_standard_error(log_returns)
print("带置信区间的夏普比率:")
print(f"  夏普比率:    {result['sharpe']:.4f}")
print(f"  标准误差:    {result['se']:.4f}")
print(f"  95% 置信区间: [{result['ci_lower']:.4f}, {result['ci_upper']:.4f}]")
print(f"  t 统计量:    {result['t_stat']:.4f}")
print(f"  p 值:        {result['p_value']:.4f}")
print(f"\n  在 5% 水平下显著? {'是 ✅' if result['p_value'] < 0.05 else '否 ❌'}")

In [ ]:
# ── 可视化：夏普 vs 样本量 ──
# 需要多少数据点才能得到可靠的估计？

sample_sizes = [100, 200, 500, 720, 1000, 1440, 2160, len(log_returns)]
results = []
for n in sample_sizes:
    if n > len(log_returns):
        continue
    subset = log_returns.iloc[-n:]
    r = sharpe_standard_error(subset)
    results.append({'n': n, 'days': n/24, **r})

res_df = pd.DataFrame(results)

fig, ax = plt.subplots(figsize=(12, 5))
ax.errorbar(res_df['days'], res_df['sharpe'], yerr=1.96*res_df['se'],
            fmt='o-', capsize=5, capthick=2, markersize=8, color='#2196F3')
ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('样本量（天）')
ax.set_ylabel('年化夏普比率')
ax.set_title('夏普比率估计 vs 样本量（含 95% 置信区间）', fontsize=14)
for _, row in res_df.iterrows():
    ax.annotate(f"n={int(row['n'])}", (row['days'], row['sharpe']),
                textcoords='offset points', xytext=(0, 12), fontsize=8, ha='center')
plt.tight_layout()
plt.show()

print("\n📌 经验法则：需要 ~2000+ 个小时样本才能获得相对紧凑的置信区间")

---

## 📐 第六节：滚动指标 — 制度敏感性

静态指标掩盖了**随时间变化的表现**。滚动指标揭示策略在不同市场制度下的行为。

In [ ]:
# ── 滚动夏普和索提诺 ──

def rolling_sharpe(returns: pd.Series, window: int = 168, periods_per_year: int = 8760) -> pd.Series:
    """滚动年化夏普比率。
    Window=168 小时 = 7 天。
    """
    rolling_mean = returns.rolling(window).mean()
    rolling_std = returns.rolling(window).std()
    return (rolling_mean / rolling_std) * np.sqrt(periods_per_year)

def rolling_sortino(returns: pd.Series, window: int = 168, periods_per_year: int = 8760) -> pd.Series:
    """滚动年化索提诺比率。"""
    rolling_mean = returns.rolling(window).mean()
    downside = returns.clip(upper=0)
    rolling_dd = np.sqrt((downside ** 2).rolling(window).mean())
    return (rolling_mean / rolling_dd) * np.sqrt(periods_per_year)

roll_sharpe = rolling_sharpe(log_returns, 168)  # 7 天窗口
roll_sortino = rolling_sortino(log_returns, 168)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 8), height_ratios=[1, 1], sharex=True)

ax1.plot(close.index, close, color='#333', linewidth=1)
ax1.set_ylabel('价格 (USDT)')
ax1.set_title('BTC/USDT 价格与滚动风险指标（7 天窗口）', fontsize=14)

ax2.plot(roll_sharpe.index, roll_sharpe, label='滚动夏普', color='#2196F3', linewidth=1)
ax2.plot(roll_sortino.index, roll_sortino, label='滚动索提诺', color='#4CAF50', linewidth=1)
ax2.axhline(0, color='gray', linestyle='--', alpha=0.5)
ax2.fill_between(roll_sharpe.index, 0, roll_sharpe,
                 where=roll_sharpe > 0, alpha=0.1, color='green')
ax2.fill_between(roll_sharpe.index, 0, roll_sharpe,
                 where=roll_sharpe < 0, alpha=0.1, color='red')
ax2.set_ylabel('年化比率')
ax2.legend()
ax2.set_ylim(-10, 10)

plt.tight_layout()
plt.show()

---

## 📐 第七节：熔断器连接

机器人的**熔断器** (`bot/risk/circuit_breaker.py`) 使用回撤作为实时风险控制：

| 级别 | 回撤 | 操作 |
|------|------|------|
| L1 | ≥ 3% | 减少仓位 |
| L2 | ≥ 5% | **停止所有交易** |

这就是为什么最大回撤和卡尔玛很重要 — 它们衡量的正是触发紧急停止的风险。

In [ ]:
# ── 熔断器模拟 ──
from bot.risk.circuit_breaker import CircuitBreaker

cb = CircuitBreaker(l1_threshold=0.03, l2_threshold=0.05)

# 在历史回撤上模拟熔断器
dd_series = compute_drawdown(log_returns)['drawdown']
statuses = [cb.check(abs(dd)) for dd in dd_series]

status_counts = pd.Series(statuses).value_counts()
print("熔断器状态分布:")
for status, count in status_counts.items():
    pct = count / len(statuses) * 100
    emoji = {'ok': '✅', 'reduce': '⚠️', 'halt': '🛑'}.get(status, '')
    print(f"  {emoji} {status:>6}: {count:>5} 个周期 ({pct:.1f}%)")

---

## 🔬 练习

### 练习 1：多资产评分卡 🔬

获取 BTC、ETH 和 SOL 的 90 天小时数据。计算每种资产的复合评分。机器人会将哪种资产排名最高？

In [ ]:
# ── 练习 1：在这里写代码 ──

# 你的代码

### 练习 2：最小追踪记录长度 ⭐

使用 Delta 方法，计算夏普比率为 1.0 在 5% 显著水平下显著所需的最少小时观测数。那是多少天？

In [ ]:
# ── 练习 2：在这里写代码 ──

# 提示：在 t_stat = SR / SE(SR) > 1.96 中求解 n
# 你的代码

---

## ✅ 知识检查

1. 夏普和索提诺的关键区别是什么？
2. 为什么机器人给索提诺 40% 的权重？
3. 回测显示夏普 = 2.5，SE = 3.0。这是好策略吗？
4. 熔断器 L2 阈值是多少？
5. 如何将小时夏普比率年化？

<details>
<summary>点击查看答案</summary>

1. 夏普惩罚所有波动率；索提诺只惩罚下行波动率
2. 加密货币收益不对称 — 不应惩罚大幅上涨，下行风险才是让账户爆仓的因素
3. 不是 — 95% CI 为 [2.5 - 5.88, 2.5 + 5.88] = [-3.38, 8.38]，包含 0。结果不具统计显著性
4. 5% 回撤 → 停止所有交易
5. 乘以 √8760（每年小时数）

</details>

---

## 🔗 下一步：Notebook 04 — 动量策略

有了数据、指标和风险度量，你已准备好构建第一个**交易策略**。在 Notebook 04 中，我们将实现动量信号引擎 — 按近期表现排名资产，并使用 RSI/EMA/成交量进行过滤。

**打开：** `04_动量策略.ipynb`